# ANEEL RAG — Etapa 2 · 2021
### Sessão paralela — processa apenas `raw_pdfs/2021/`

> Rode em paralelo com os notebooks de 2016 e 2022.
> Se a sessão cair, reexecute a Célula 8 — retoma via checkpoint.

**Saída:**
- `extracted/2021/doc_id.md`
- `extracted/2021/doc_id.meta.json`
- `logs/gate_extraction_2021.csv`

## Célula 1 — Dependências

In [1]:
!pip install docling pymupdf pdf2image rouge-score jiwer --quiet
!apt-get install -y poppler-utils --quiet
import torch
print('Sessao: 2021')
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  {torch.cuda.get_device_name(0)}')
    print(f'  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.0/494.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 152.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Célula 2 — Drive e estrutura

In [2]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd, re, json, time, signal

BASE = Path('/content/drive/MyDrive/aneel_rag')
ANO  = '2021'

DIRS = {
    'raw_pdfs':  BASE / 'raw_pdfs'  / ANO,
    'extracted': BASE / 'extracted' / ANO,
    'logs':      BASE / 'logs',
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

n = len(list(DIRS['raw_pdfs'].glob('*.pdf')))
print(f'Sessao 2021: {n:,} PDFs')

Mounted at /content/drive
Sessao 2021: 9,517 PDFs


## Célula 3 — Verificar PDFs

In [3]:
import zipfile
from tqdm.auto import tqdm

zip_path = BASE / 'pdf.zip'

if not zip_path.exists():
    print(f'Nao encontrado: {zip_path}')
else:
    print(f'ZIP: {zip_path.stat().st_size/(1024**3):.2f} GB')
    with zipfile.ZipFile(zip_path, 'r') as z:
        members = z.namelist()
        print(f'  Arquivos totais no ZIP: {len(members):,}')
        # Detectar prefixo raiz (ex: pdf/)
        prefixo = next((m for m in members if m.endswith('/')), '')
        # Filtrar apenas arquivos do ANO desta sessao
        membros_ano = [
            m for m in members
            if not m.endswith('/')
            and m.startswith(f'{prefixo}{ANO}/')
        ]
        print(f'  Arquivos de {ANO}: {len(membros_ano):,}')
        novos = skipped = 0
        for member in tqdm(membros_ano, desc=f'Extraindo {ANO}', unit='arq'):
            nome = member[len(f'{prefixo}{ANO}/'):]
            if not nome: continue
            dest = DIRS['raw_pdfs'] / nome
            if dest.exists(): skipped += 1; continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            with z.open(member) as src, open(dest,'wb') as dst:
                dst.write(src.read())
            novos += 1
    n_total = len(list(DIRS['raw_pdfs'].glob('*.pdf')))
    print(f'  Novos   : {novos:,}')
    print(f'  Existiam: {skipped:,}')
    print(f'  Total em raw_pdfs/{ANO}/: {n_total:,} PDFs')

ZIP: 3.70 GB
  Arquivos totais no ZIP: 27,281
  Arquivos de 2021: 9,517


Extraindo 2021:   0%|          | 0/9517 [00:00<?, ?arq/s]

  Novos   : 0
  Existiam: 9,517
  Total em raw_pdfs/2021/: 9,517 PDFs


## Célula 4 — Configurações e métricas

| Categoria | Peso |
|---|---|
| Conteúdo (densidade, comprimento, chars/pag, palavras rec.) | 45% |
| Estrutura (headings, artigos, tabelas) | 30% |
| Cobertura (pags com texto / total) | 25% |
| Referência (ROUGE-L, CER, WER — opcional) | substitui cobertura |

In [4]:
from rouge_score import rouge_scorer
from jiwer import cer as compute_cer, wer as compute_wer
import fitz

CHECKPOINT_EVERY = 50
LOG_PATH = DIRS['logs'] / 'gate_extraction_2021.csv'

GATE = {
    'min_score':         0.50,
    'min_densidade':     0.60,
    'min_chars_pag':     80,
    'min_palavras_rec':  0.50,
    'timeout_normal_s':  120,
    'timeout_grande_s':  600,
    'limite_sem_tab_kb': 2048,
}

_vocab = (
    'de do da dos das em no na nos nas para por com um uma o a os as '
    'que se ao aos pela pelo pelos pelas este esta estes estas esse '
    'essa esses essas nao sim art artigo paragrafo inciso lei decreto '
    'resolucao portaria despacho publicado federal nacional diario '
    'oficial agencia energia eletrica aneel considerando resolve '
    'determina estabelece dispoe processo interessado numero data '
    'autorizacao concessao permissao servico sistema ministerio '
    'estado ministro diretor superintendente ser estar ter fazer '
)
VOCAB_PTBR = set(_vocab.split())
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def clean_text(t):
    t = re.sub(r'\n{3,}', '\n\n', t)
    return re.sub(r' {2,}', ' ', t).strip()

def is_valid_pdf(path):
    try:
        with open(path,'rb') as f: return f.read(4)==b'%PDF'
    except: return False

def precisa_ocr(pdf_path):
    try:
        doc = fitz.open(str(pdf_path))
        for page in doc:
            if len(page.get_text().strip()) > 20:
                doc.close(); return False
        doc.close(); return True
    except: return True

def contar_paginas_com_texto(pdf_path):
    try:
        doc = fitz.open(str(pdf_path))
        total = len(doc)
        com_texto = sum(1 for p in doc if len(p.get_text().strip()) > 10)
        doc.close(); return com_texto, total
    except: return 0, 0

def metrica_conteudo(text, num_pages):
    if not text or not text.strip():
        return {'densidade':0.0,'comprimento':0,'chars_por_pag':0.0,'palavras_rec':0.0,'score':0.0}
    cv  = sum(1 for c in text if c.isalnum() or c in ' .,;:\n-()/')
    den = cv / len(text)
    cpp = len(text) / max(num_pages, 1)
    tok = re.findall(r'[a-zA-ZaeiouáéíóúâêôàãõçÁÉÍÓÚÂÊÔÀÃÕÇ]{3,}', text.lower())
    pr  = sum(1 for t in tok if t in VOCAB_PTBR) / max(len(tok), 1)
    sc  = (0.30*min(den/GATE['min_densidade'],1) + 0.20*min(len(text)/500,1)
           + 0.30*min(cpp/GATE['min_chars_pag'],1) + 0.20*min(pr/GATE['min_palavras_rec'],1))
    return {'densidade':round(den,3),'comprimento':len(text),
            'chars_por_pag':round(cpp,1),'palavras_rec':round(pr,3),'score':round(sc,3)}

def metrica_estrutura(text, num_tables):
    if not text: return {'n_headings':0,'n_artigos':0,'n_tabelas':0,'score':0.0}
    nh = len(re.findall(r'^#{1,4}\s', text, re.MULTILINE))
    na = len(re.findall(r'Art\.?\s*\d+|ARTIGO\s+\d+|Par[aá]grafo\s+[Uu]n', text, re.IGNORECASE))
    sc = (0.5 if (nh==0 and na==0)
          else min(0.40*min(nh/3,1)+0.40*min(na/2,1)+0.20*(1 if num_tables>0 else 0), 1))
    return {'n_headings':nh,'n_artigos':na,'n_tabelas':num_tables,'score':round(sc,3)}

def metrica_cobertura(pdf_path, num_pages):
    pt,total = contar_paginas_com_texto(pdf_path)
    if total==0: return {'pags_com_texto':0,'total_pags':0,'cobertura':1.0,'score':1.0}
    cob = pt/total
    return {'pags_com_texto':pt,'total_pags':total,'cobertura':round(cob,3),'score':round(cob,3)}

def metrica_referencia(text, ground_truth):
    if not ground_truth or not text:
        return {'rouge_l':None,'cer':None,'wer':None,'score':None,'nota':'sem ground_truth'}
    try:
        rl = rouge.score(ground_truth[:1000], text[:1000])['rougeL'].fmeasure
        cv = compute_cer(ground_truth, text)
        wv = compute_wer(ground_truth, text)
        sc = 0.40*rl + 0.30*max(0,1-cv) + 0.30*max(0,1-wv)
        return {'rouge_l':round(rl,4),'cer':round(cv,4),'wer':round(wv,4),'score':round(sc,3),'nota':'ok'}
    except Exception as e:
        return {'rouge_l':None,'cer':None,'wer':None,'score':None,'nota':str(e)[:80]}

def avaliar_qualidade(text, pdf_path, num_pages, num_tables, ground_truth=None):
    c=metrica_conteudo(text,num_pages); e=metrica_estrutura(text,num_tables)
    v=metrica_cobertura(pdf_path,num_pages); r=metrica_referencia(text,ground_truth)
    sf = (0.35*c['score']+0.25*e['score']+0.20*v['score']+0.20*r['score']
          if r['score'] else 0.45*c['score']+0.30*e['score']+0.25*v['score'])
    return {'score_final':round(sf,3),'gate_passou':sf>=GATE['min_score'],
            'conteudo':c,'estrutura':e,'cobertura':v,'referencia':r}

def load_existing_log():
    if LOG_PATH.exists():
        df = pd.read_csv(LOG_PATH)
        ok = set(df[df['status'].str.startswith('ok')]['doc_id'].tolist())
        print(f'Log 2021: {len(ok):,} ja extraidos')
        return ok
    return set()

print('Config 2021 carregada')
print(f'  score_min={GATE["min_score"]}  timeout_normal={GATE["timeout_normal_s"]}s  timeout_grande={GATE["timeout_grande_s"]}s')

Config 2021 carregada
  score_min=0.5  timeout_normal=120s  timeout_grande=600s


## Célula 5 — Docling com OCR automático e timeout

In [5]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

class DoclingTimeout(Exception): pass
def _alarm_handler(signum, frame): raise DoclingTimeout()

def make_converter(com_ocr, com_tabelas=True):
    opts = PdfPipelineOptions()
    opts.do_ocr             = com_ocr
    opts.do_table_structure = com_tabelas
    if com_tabelas: opts.table_structure_options.do_cell_matching = True
    return DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)})

converter_digital     = make_converter(False, True)
converter_ocr         = make_converter(True,  True)
converter_sem_tabelas = make_converter(False, False)

def extract_docling(pdf_path):
    r  = {'text':'','markdown':'','status':'ok','error':'','ocr_usado':False,'num_pages':0,'num_tables':0}
    kb = pdf_path.stat().st_size // 1024
    to = GATE['timeout_grande_s'] if kb > 1024 else GATE['timeout_normal_s']
    try:
        usa_ocr   = precisa_ocr(pdf_path)
        conv_usar = (converter_sem_tabelas if kb > GATE['limite_sem_tab_kb']
                     else converter_ocr if usa_ocr else converter_digital)
        signal.signal(signal.SIGALRM, _alarm_handler)
        signal.alarm(to)
        try:
            conv = conv_usar.convert(str(pdf_path))
            signal.alarm(0)
        except DoclingTimeout:
            r['status']='timeout'; r['error']=f'timeout {to}s ({kb}KB)'; return r
        doc = conv.document
        r['markdown']   = clean_text(doc.export_to_markdown())
        r['text']       = clean_text(doc.export_to_text())
        r['ocr_usado']  = usa_ocr
        r['num_pages']  = int(doc.num_pages) if hasattr(doc,'num_pages') else 0
        r['num_tables'] = len(doc.tables) if hasattr(doc,'tables') else 0
        if not r['text'].strip(): r['status']='empty'; r['error']='texto vazio'
    except Exception as e:
        signal.alarm(0); r['status']='error'; r['error']=str(e)[:200]
    return r

print('Docling pronto')
print('  digital: OCR=False TableFormer=True')
print('  ocr    : OCR=True  TableFormer=True')
print(f'  sem_tab: OCR=False TableFormer=False  (PDFs > {GATE["limite_sem_tab_kb"]}KB)')

Docling pronto
  digital: OCR=False TableFormer=True
  ocr    : OCR=True  TableFormer=True
  sem_tab: OCR=False TableFormer=False  (PDFs > 2048KB)


## Célula 6 — Fallback VLM Qwen2.5-VL 7B

In [6]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
import torch
from pdf2image import convert_from_path
VLM_MODEL=VLM_PROCESSOR=None; VLM_LOADED=False

def load_vlm():
    global VLM_MODEL,VLM_PROCESSOR,VLM_LOADED
    if VLM_LOADED: return
    print('Carregando Qwen2.5-VL-7B...')
    VLM_MODEL = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        'Qwen/Qwen2.5-VL-7B-Instruct',torch_dtype=torch.float16,device_map='auto')
    VLM_PROCESSOR = AutoProcessor.from_pretrained('Qwen/Qwen2.5-VL-7B-Instruct')
    VLM_LOADED=True; print('VLM pronto')

def extract_vlm(pdf_path):
    r={'text':'','markdown':'','status':'ok','error':'','ocr_usado':True,'num_pages':0,'num_tables':0}
    try:
        load_vlm()
        pages=convert_from_path(str(pdf_path),dpi=150); r['num_pages']=len(pages)
        texts=[]
        for i,page in enumerate(pages[:20]):
            from qwen_vl_utils import process_vision_info
            msgs=[{'role':'user','content':[
                {'type':'image','image':page},
                {'type':'text','text':'Extraia todo o texto desta pagina de legislacao brasileira. Preserve artigos, paragrafos e tabelas. Retorne apenas o texto.'}
            ]}]
            prompt=VLM_PROCESSOR.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
            img_in,_=process_vision_info(msgs)
            inputs=VLM_PROCESSOR(text=[prompt],images=img_in,return_tensors='pt').to('cuda')
            with torch.no_grad(): ids=VLM_MODEL.generate(**inputs,max_new_tokens=2048)
            out=VLM_PROCESSOR.batch_decode([ids[0][inputs.input_ids.shape[1]:]],skip_special_tokens=True)[0]
            texts.append(f'## Pagina {i+1}\n\n{out}')
        full='\n\n'.join(texts)
        r['markdown']=clean_text(full); r['text']=clean_text(re.sub(r'#+\s*','',full))
        if not r['text'].strip(): r['status']='empty'; r['error']='VLM vazio'
    except Exception as e: r['status']='error'; r['error']=str(e)[:200]
    return r

print('VLM definido (lazy loading)')

VLM definido (lazy loading)


## Célula 7 — Validação com 5 PDFs (2021)

> Execute sempre antes da produção.
> Testa pipeline + métricas + timeout com PDFs reais.

In [7]:
GROUND_TRUTH = None

# ── Fix 1: contar paginas via PyMuPDF ─────────────────────────────────────
def get_num_pages_fitz(pdf_path) -> int:
    try:
        doc = fitz.open(str(pdf_path))
        n   = len(doc)
        doc.close()
        return max(n, 1)
    except:
        return 1

# ── Fix 2: vocabulario expandido ──────────────────────────────────────────
_vocab_aneel = (
    'agencia nacional energia eletrica aneel resolucao normativa '
    'despacho portaria autorizativa superintendencia gerencia '
    'megawatt quilowatt kilovolt mwh kwh mw kv gw tensao '
    'transmissao distribuicao geracao comercializacao '
    'concessao permissao autorizacao outorga licitacao '
    'contrato tarifa consumidor usuario titular agente '
    'instalacao subestacao linha usina hidreletrica '
    'termoeletrica eolica fotovoltaica potencia capacidade '
    'regulatorio regulamentacao fiscalizacao homologacao '
    'interligado sistema isolado ons anp ibama '
    'concessionaria distribuidora geradora transmissora '
    'requerente requerida interessado interessada processo '
    'protocolo oficio circular instrucao normativa '
)
VOCAB_PTBR.update(_vocab_aneel.split())

# ── Fix 3: metricas rebalanceadas ─────────────────────────────────────────
GATE['min_score'] = 0.80

def metrica_conteudo(text, num_pages):
    if not text or not text.strip():
        return {'densidade':0.0,'comprimento':0,'chars_por_pag':0.0,'palavras_rec':0.0,'score':0.0}
    cv  = sum(1 for c in text if c.isalnum() or c in ' .,;:\n-()/')
    den = cv / len(text)
    cpp = len(text) / max(int(num_pages) if num_pages else 1, 1)
    tok = re.findall(r'[a-zA-ZaeiouáéíóúâêôàãõçÁÉÍÓÚÂÊÔÀÃÕÇ]{3,}', text.lower())
    pr  = sum(1 for t in tok if t in VOCAB_PTBR) / max(len(tok), 1)
    sc  = (0.45 * min(den / GATE['min_densidade'], 1.0) +
           0.30 * min(len(text) / 300, 1.0) +
           0.15 * min(cpp / GATE['min_chars_pag'], 1.0) +
           0.10 * min(pr  / GATE['min_palavras_rec'], 1.0))
    return {'densidade':round(den,3),'comprimento':len(text),
            'chars_por_pag':round(cpp,1),'palavras_rec':round(pr,3),'score':round(sc,3)}

def metrica_estrutura(text, num_tables):
    if not text:
        return {'n_headings':0,'n_artigos':0,'n_tabelas':0,'score':0.0}
    nh = len(re.findall(r'^#{1,4}\s', text, re.MULTILINE))
    na = len(re.findall(r'Art\.?\s*\d+|ARTIGO\s+\d+|Par[aá]grafo\s+[Uu]n', text, re.IGNORECASE))
    if nh == 0 and na == 0:
        sc = 0.70
    elif nh == 0 and na > 0:
        sc = 0.75 + 0.10 * min(na / 3, 1.0)
    else:
        sc = min(0.40*min(nh/3,1)+0.40*min(na/2,1)+0.20*(1 if num_tables>0 else 0), 1.0)
    return {'n_headings':nh,'n_artigos':na,'n_tabelas':num_tables,'score':round(sc,3)}

def avaliar_qualidade(text, pdf_path, num_pages, num_tables, ground_truth=None):
    np_real = get_num_pages_fitz(pdf_path)
    c = metrica_conteudo(text, np_real)
    e = metrica_estrutura(text, num_tables)
    v = metrica_cobertura(pdf_path, np_real)
    r = metrica_referencia(text, ground_truth)
    if r['score'] is not None:
        sf = 0.45*c['score'] + 0.10*e['score'] + 0.20*v['score'] + 0.25*r['score']
    else:
        sf = 0.55*c['score'] + 0.15*e['score'] + 0.30*v['score']
    return {'score_final':round(sf,3),'gate_passou':sf>=GATE['min_score'],
            'conteudo':c,'estrutura':e,'cobertura':v,'referencia':r}

def extract_docling(pdf_path):
    r = {'text':'','markdown':'','status':'ok','error':'','ocr_usado':False,'num_pages':0,'num_tables':0}
    kb = pdf_path.stat().st_size // 1024
    to = GATE['timeout_grande_s'] if kb > 1024 else GATE['timeout_normal_s']
    try:
        usa_ocr   = precisa_ocr(pdf_path)
        conv_usar = (converter_sem_tabelas if kb > GATE['limite_sem_tab_kb']
                     else converter_ocr if usa_ocr else converter_digital)
        signal.signal(signal.SIGALRM, _alarm_handler)
        signal.alarm(to)
        try:
            conv = conv_usar.convert(str(pdf_path))
            signal.alarm(0)
        except DoclingTimeout:
            r['status']='timeout'; r['error']=f'timeout {to}s ({kb}KB)'; return r
        doc = conv.document
        r['markdown']   = clean_text(doc.export_to_markdown())
        r['text']       = clean_text(doc.export_to_text())
        r['ocr_usado']  = usa_ocr
        r['num_pages']  = get_num_pages_fitz(pdf_path)
        r['num_tables'] = len(doc.tables) if hasattr(doc,'tables') else 0
        if not r['text'].strip(): r['status']='empty'; r['error']='texto vazio'
    except Exception as e:
        signal.alarm(0)
        if r['text'].strip():
            r['error'] = str(e)[:100]
        else:
            r['status'] = 'error'; r['error'] = str(e)[:200]
    return r

print(f'Ajustes: vocab={len(VOCAB_PTBR)} termos | min_score={GATE["min_score"]}')
print('Pesos: conteudo=55% estrutura=15% cobertura=30%')

# ── Validacao ─────────────────────────────────────────────────────────────
validos = [p for p in sorted(DIRS['raw_pdfs'].glob('*.pdf')) if is_valid_pdf(p)]
if not validos:
    print('Nenhum PDF valido encontrado')
else:
    # Excluir PDFs > 10MB da amostra de validacao — foco nos docs normais
    validos_val = [p for p in validos if p.stat().st_size < 10*1024*1024]
    peq = sorted(validos_val, key=lambda p: p.stat().st_size)[:2]
    grd = sorted(validos_val, key=lambda p: p.stat().st_size, reverse=True)[:2]
    med = [validos_val[len(validos_val)//2]]
    amostra = list(dict.fromkeys(peq+med+grd))[:5]
    print('='*65); print(f'VALIDACAO {ANO} — {len(amostra)} PDFs (excluindo >10MB)'); print('='*65)
    resultados = []

    for i,pdf_path in enumerate(amostra):
        kb = pdf_path.stat().st_size // 1024
        print(f'\n[{i+1}/{len(amostra)}] {pdf_path.name[:52]:<52} ({kb:>6} KB)')
        t0=time.time(); usa_ocr=precisa_ocr(pdf_path)
        sem_tab = kb > GATE['limite_sem_tab_kb']
        print(f'  OCR:{"SIM" if usa_ocr else "NAO"}  TableFormer:{"NAO" if sem_tab else "SIM"}')
        r=extract_docling(pdf_path); te=time.time()-t0
        status_display = r['status'] if not r['error'] else f'{r["status"]}*'
        print(f'  Docling:{status_display:<10} {len(r["text"]):>8,} chars {te:.1f}s')
        if r['error'] and r['status']=='ok':
            print(f'  Aviso: {r["error"][:80]}')
        qa=avaliar_qualidade(r['text'],pdf_path,r['num_pages'],r['num_tables'],GROUND_TRUTH)
        c,e,v,ref=qa['conteudo'],qa['estrutura'],qa['cobertura'],qa['referencia']
        print(f'  Score:{qa["score_final"]:.3f} ({"PASSOU" if qa["gate_passou"] else "FALHOU"}) [min={GATE["min_score"]}]')
        print(f'  Cont:{c["score"]:.2f} den={c["densidade"]:.2f} cpp={c["chars_por_pag"]:.0f} pal={c["palavras_rec"]:.2f}')
        print(f'  Estr:{e["score"]:.2f} hdg={e["n_headings"]} art={e["n_artigos"]} tab={e["n_tabelas"]}')
        print(f'  Cob:{v["score"]:.2f} {v["pags_com_texto"]}/{v["total_pags"]} pags')
        if ref['score'] is not None:
            print(f'  Ref:{ref["score"]:.2f} rouge={ref["rouge_l"]:.3f} cer={ref["cer"]:.3f} wer={ref["wer"]:.3f}')
        extrator='docling'; md_f=r['markdown']; fb=False
        # VLM fallback apenas para PDFs < 5MB — inviavel em docs maiores
        if not qa['gate_passou'] and r['status'] not in ('timeout','error') and kb < 5000:
            print('  -> VLM fallback...')
            rv=extract_vlm(pdf_path)
            if rv['status']=='ok':
                qav=avaliar_qualidade(rv['text'],pdf_path,rv['num_pages'],0,GROUND_TRUTH)
                if qav['gate_passou']:
                    extrator='vlm_qwen'; md_f=rv['markdown']; fb=True; qa=qav
                    print(f'  -> VLM ok:{qav["score_final"]:.3f}')
                else:
                    extrator='docling_degradado'
                    print(f'  -> VLM falhou:{qav["score_final"]:.3f}')
        elif not qa['gate_passou'] and kb >= 5000:
            print(f'  -> Score baixo mas PDF grande ({kb}KB) — usando Docling mesmo assim')
        resultados.append({'arquivo':pdf_path.name,'kb':kb,'extrator':extrator,
            'score':qa['score_final'],'tempo_s':round(time.time()-t0,1),'fallback':fb})

    df_v=pd.DataFrame(resultados)
    total_ano=len(list(DIRS['raw_pdfs'].glob('*.pdf')))
    df_normais = df_v[df_v['kb'] < 5000]
    t_med = df_normais['tempo_s'].mean() if len(df_normais) else df_v['tempo_s'].mean()
    horas_est = (t_med * total_ano * 0.994 + 200 * total_ano * 0.006) / 3600
    print('='*65)
    print(f'VALIDACAO {ANO}: score={df_v["score"].mean():.3f} gate_min={GATE["min_score"]}')
    print(f'Tempo medio (sem outliers >5MB): {t_med:.1f}s')
    print(f'Estimativa {ANO}: {horas_est:.1f}h')
    df_v.to_csv(DIRS['logs']/f'validacao_{ANO}.csv',index=False)
    n_ok=(df_v['extrator']!='docling_degradado').sum()
    status_txt = 'APROVADA' if n_ok>=4 else 'ATENCAO'
    acao_txt   = 'Avance para Celula 8.' if n_ok>=4 else 'Revise os scores antes da producao.'
    print(f'{status_txt} ({n_ok}/5) — {acao_txt}')

Ajustes: vocab=133 termos | min_score=0.8
Pesos: conteudo=55% estrutura=15% cobertura=30%
VALIDACAO 2021 — 5 PDFs (excluindo >10MB)

[1/5] 2021__area202110954_1.pdf                            (    22 KB)
  OCR:SIM  TableFormer:SIM


[INFO] 2026-04-18 14:16:39,384 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-18 14:16:39,388 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-18 14:16:39,390 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-18 14:16:40,469 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-04-18 14:16:40,878 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-18 14:16:40,880 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-18 14:16:41,470 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-18 14:16:41,472 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-18 14:16:41,473 [RapidOCR] download_file.py:68: Init

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

  Docling:empty*            0 chars 13.1s
  Score:0.000 (FALHOU) [min=0.8]
  Cont:0.00 den=0.00 cpp=0 pal=0.00
  Estr:0.00 hdg=0 art=0 tab=0
  Cob:0.00 0/1 pags
  -> VLM fallback...
Carregando Qwen2.5-VL-7B...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

VLM pronto

[2/5] 2021__dsp2021522.pdf                                 (    44 KB)
  OCR:NAO  TableFormer:SIM


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

  Docling:ok              698 chars 1.4s
  Score:0.931 (PASSOU) [min=0.8]
  Cont:0.96 den=1.00 cpp=698 pal=0.29
  Estr:0.70 hdg=0 art=0 tab=0
  Cob:1.00 1/1 pags

[3/5] 2021__dsp20213106ti.pdf                              (    71 KB)
  OCR:NAO  TableFormer:SIM
  Docling:ok              821 chars 0.6s
  Score:0.929 (PASSOU) [min=0.8]
  Cont:0.95 den=1.00 cpp=821 pal=0.26
  Estr:0.70 hdg=0 art=0 tab=0
  Cob:1.00 1/1 pags

[4/5] 2021__nreh20212886_1.pdf                             (  8900 KB)
  OCR:NAO  TableFormer:NAO


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

  Docling:ok          152,371 chars 26.4s
  Score:0.940 (PASSOU) [min=0.8]
  Cont:0.93 den=0.92 cpp=3047 pal=0.15
  Estr:0.85 hdg=0 art=22 tab=30
  Cob:1.00 50/50 pags

[5/5] 2021__nreh20212862.pdf                               (  7428 KB)
  OCR:NAO  TableFormer:NAO
  Docling:ok          123,055 chars 21.6s
  Score:0.941 (PASSOU) [min=0.8]
  Cont:0.93 den=0.86 cpp=2511 pal=0.17
  Estr:0.85 hdg=0 art=30 tab=28
  Cob:1.00 49/49 pags
VALIDACAO 2021: score=0.748 gate_min=0.8
Tempo medio (sem outliers >5MB): 24.1s
Estimativa 2021: 66.5h
APROVADA (5/5) — Avance para Celula 8.


## Célula 8 — Produção: todos os PDFs de 2021

> Só execute após VALIDAÇÃO APROVADA.
> Checkpoint a cada 50 docs — retoma automaticamente se a sessão cair.

In [ ]:
def processar_pdf(pdf_path):
    doc_id=pdf_path.stem
    kb=pdf_path.stat().st_size//1024
    log={'doc_id':doc_id,'arquivo':pdf_path.name,'ano':ANO,'status':'erro',
         'extrator':'','score_final':0.0,'ocr_usado':False,'chars':0,
         'num_pages':0,'num_tables':0,'tempo_s':0.0,'erro':''}
    if not is_valid_pdf(pdf_path):
        log['erro']='magic bytes invalidos'; return log,None,None
    t0=time.time(); r=extract_docling(pdf_path)
    if r['status']=='timeout':
        log.update({'status':'timeout','erro':r['error'],'tempo_s':round(time.time()-t0,1)})
        return log, r.get('markdown',''), {'doc_id':doc_id,'timeout':True,'arquivo':pdf_path.name,'ano':ANO}
    qa=avaliar_qualidade(r['text'],pdf_path,r['num_pages'],r['num_tables'])
    md_f=r['markdown']; extrator='docling'
    if not qa['gate_passou'] and kb < 5000:
        rv=extract_vlm(pdf_path)
        if rv['status']=='ok':
            qav=avaliar_qualidade(rv['text'],pdf_path,rv['num_pages'],0)
            if qav['gate_passou']: extrator='vlm_qwen'; md_f=rv['markdown']; qa=qav
            else: extrator='docling_degradado'
        else: extrator='docling_degradado'
    elif not qa['gate_passou'] and kb >= 5000:
        extrator='docling_degradado'
    sm={'docling':'ok_gate','vlm_qwen':'ok_fallback'}
    log.update({'status':sm.get(extrator,'ok_degradado'),'extrator':extrator,
        'score_final':qa['score_final'],'ocr_usado':r['ocr_usado'],
        'chars':len(md_f),'num_pages':r['num_pages'],'num_tables':r['num_tables'],
        'tempo_s':round(time.time()-t0,1)})
    meta={'doc_id':doc_id,'arquivo':pdf_path.name,'ano':ANO,'extrator':extrator,
        'score_final':qa['score_final'],'gate_passou':qa['gate_passou'],
        'ocr_usado':r['ocr_usado'],'chars':len(md_f),
        'num_pages':r['num_pages'],'num_tables':r['num_tables'],
        'metricas':{'conteudo':qa['conteudo'],'estrutura':qa['estrutura'],'cobertura':qa['cobertura']}}
    return log, md_f, meta


def run_extraction():
    from tqdm.auto import tqdm
    ja_ok=load_existing_log()
    todos=[p for p in sorted(DIRS['raw_pdfs'].glob('*.pdf'))
           if p.stem not in ja_ok and is_valid_pdf(p)]
    total_ano=len(list(DIRS['raw_pdfs'].glob('*.pdf')))
    print(f'Ano {ANO}: total={total_ano:,} | ja_ok={len(ja_ok):,} | pendentes={len(todos):,}')
    if not todos: print('Todos ja extraidos!'); return
    log_buf=[]; ok=fb=deg=err=tmo=0; t0=time.time()
    with tqdm(total=len(todos),desc=f'Extracao {ANO}',unit='doc',dynamic_ncols=True) as pb:
        for pdf_path in todos:
            le,md,meta=processar_pdf(pdf_path)
            if md is not None:
                (DIRS['extracted']/f'{pdf_path.stem}.md').write_text(md,encoding='utf-8')
                (DIRS['extracted']/f'{pdf_path.stem}.meta.json').write_text(
                    json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
            log_buf.append(le)
            s=le['status']
            if   s=='ok_gate':     ok+=1
            elif s=='ok_fallback': fb+=1
            elif s=='ok_degradado':deg+=1
            elif s=='timeout':     tmo+=1
            else:                  err+=1
            pb.set_postfix({'ok':ok,'vlm':fb,'tmo':tmo,'err':err}); pb.update(1)
            if len(log_buf)>=CHECKPOINT_EVERY:
                pd.DataFrame(log_buf).to_csv(LOG_PATH,mode='a',header=not LOG_PATH.exists(),index=False)
                log_buf.clear()
    if log_buf: pd.DataFrame(log_buf).to_csv(LOG_PATH,mode='a',header=not LOG_PATH.exists(),index=False)
    h=(time.time()-t0)/3600
    print(f'\nAno {ANO} concluido em {h:.1f}h')
    print(f'  OK:{ok:,}  VLM:{fb:,}  Deg:{deg:,}  Timeout:{tmo:,}  Err:{err:,}')


run_extraction()

Ano 2021: total=9,517 | ja_ok=0 | pendentes=9,517


Extracao 2021:   0%|          | 0/9517 [00:00<?, ?doc/s]

## Célula 9 — Relatório final 2021

In [ ]:
if not LOG_PATH.exists():
    print('Log nao existe — execute a Celula 8 primeiro')
else:
    df=pd.read_csv(LOG_PATH)
    print(f'RELATORIO FINAL 2021 | Total:{len(df):,}')
    for s,n in df['status'].value_counts().items():
        print(f'  {s:<22}: {n:>6,} ({100*n/len(df):.1f}%)')
    ok_df=df[df['status'].str.startswith('ok')]
    if len(ok_df):
        print(f'Score medio  : {ok_df["score_final"].mean():.3f}')
        print(f'Alta qualidade (>=0.70): {(ok_df["score_final"]>=0.70).mean()*100:.1f}%')
        print(f'Chars medio  : {ok_df["chars"].mean():,.0f}')
    t_df=df[df['status']=='timeout']
    if len(t_df): print(f'Timeouts     : {len(t_df)}')
    e_df=df[df['status']=='erro']
    if len(e_df):
        e_df.to_csv(DIRS['logs']/f'erros_{ANO}.csv',index=False)
        print(f'Erros        : {len(e_df)} — salvos em erros_{ANO}.csv')
    print(f'Markdowns em : {DIRS["extracted"]}')
    print(f'Proxima etapa: Etapa 3 — Chunking hierarquico L0/L1/L2')